In [ ]:
# Setup for Matchett Human dataset

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from importlib.metadata import version

# Scanpy settings
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=150, facecolor='white', figsize=(8, 6))

# Paths
from pathlib import Path
base_path = os.environ.get("TRAJRISK_ROOT") or str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "metadata" / "source_provenance.json").is_file()))
data_raw = os.path.join(base_path, "Data/raw")
data_processed = os.path.join(base_path, "Data/processed")
results_path = os.path.join(base_path, "results")

matchett_human_path = os.path.join(data_raw, "matchett_human_snrna_GSE223581")

print(" Setup complete for Matchett Human dataset!")
print(f"Scanpy version: {version('scanpy')}")

# List files
print("\n Files in directory:")
if os.path.exists(matchett_human_path):
    files = [f for f in os.listdir(matchett_human_path) if not f.startswith('.')]
    for f in sorted(files):
        file_path = os.path.join(matchett_human_path, f)
        if os.path.isfile(file_path):
            size = os.path.getsize(file_path) / 1e6
            print(f"   {f} ({size:.1f} MB)")
        elif os.path.isdir(file_path):
            print(f"   {f}/ (directory)")
else:
    print(f"     Directory not found: {matchett_human_path}")

In [ ]:
# Load Matchett Human dataset (APAP + Healthy only)

import glob

# Get all MTX files
all_mtx = sorted(glob.glob(os.path.join(matchett_human_path, "*_matrix.mtx.gz")))

# Filter for APAP and Healthy only (skip nae samples)
apap_samples = [f for f in all_mtx if 'apap' in os.path.basename(f)]
healthy_samples = [f for f in all_mtx if 'hea' in os.path.basename(f)]

selected_samples = apap_samples + healthy_samples

print(f" Found {len(apap_samples)} APAP samples")
print(f" Found {len(healthy_samples)} Healthy samples")
print(f" Total samples to load: {len(selected_samples)}")
print(f" Skipping {len(all_mtx) - len(selected_samples)} non-APAP ALF samples\n")

adatas = []

for i, mtx_file in enumerate(selected_samples, 1):
    # Get sample ID from filename
    sample_id = os.path.basename(mtx_file).replace('_matrix.mtx.gz', '')
    sample_dir = os.path.dirname(mtx_file)
    
    print(f"[{i}/{len(selected_samples)}] Loading {sample_id}...")
    
    # Read 10x MTX format
    adata = sc.read_10x_mtx(
        sample_dir,
        var_names='gene_symbols',
        prefix=sample_id + '_',
        gex_only=True
    )
    
    # Make gene names unique
    adata.var_names_make_unique()
    
    print(f"      Raw: {adata.n_obs:>8,} cells")
    
    # Fast filter: >200 genes
    genes_per_cell = (adata.X > 0).sum(axis=1).A1
    cell_filter = genes_per_cell > 200
    adata = adata[cell_filter, :].copy()
    
    print(f"      Filtered: {adata.n_obs:>6,} cells (>200 genes)")
    
    # Add unique cell IDs
    adata.obs_names = [f"{barcode}_{sample_id}" for barcode in adata.obs_names]
    
    # Add metadata
    adata.obs['sample'] = sample_id
    
    # Determine condition
    if 'apap' in sample_id:
        adata.obs['condition'] = 'APAP_ALF'
        adata.obs['patient_type'] = 'APAP'
    else:  # hea
        adata.obs['condition'] = 'Healthy'
        adata.obs['patient_type'] = 'Healthy'
    
    adatas.append(adata)
    print(f"       Done\n")

print("="*80)

# Concatenate
print(" Concatenating all samples...")
adata_matchett = sc.concat(adatas, join='outer', index_unique=None)

print("\n Dataset loaded!")
print(f" Total cells: {adata_matchett.n_obs:,}")
print(f" Total genes: {adata_matchett.n_vars:,}")

print(f"\n Cells per condition:")
print(adata_matchett.obs['condition'].value_counts())

print(f"\n Cells per sample:")
sample_counts = adata_matchett.obs['sample'].value_counts()
for sample, count in sorted(sample_counts.items()):
    print(f"   {sample}: {count:>6,} cells")

In [ ]:
# Calculate QC metrics

# Identify mitochondrial genes (HUMAN genes start with MT-, not mt-)
adata_matchett.var['mt'] = adata_matchett.var_names.str.startswith('MT-')
n_mt = adata_matchett.var['mt'].sum()
print(f" Found {n_mt} mitochondrial genes")

# Calculate QC
sc.pp.calculate_qc_metrics(
    adata_matchett,
    qc_vars=['mt'],
    percent_top=None,
    log1p=False,
    inplace=True
)

print("\n QC METRICS:")
print("="*80)
print(f" Median genes/cell: {adata_matchett.obs['n_genes_by_counts'].median():.0f}")
print(f" Median UMI/cell: {adata_matchett.obs['total_counts'].median():.0f}")
print(f" Median mito %: {adata_matchett.obs['pct_counts_mt'].median():.2f}%")

print("\n BY CONDITION:")
for condition in ['Healthy', 'APAP_ALF']:
    subset = adata_matchett.obs[adata_matchett.obs['condition'] == condition]
    print(f"\n{condition}: {len(subset):,} cells")
    print(f"  Median genes: {subset['n_genes_by_counts'].median():.0f}")
    print(f"  Median UMI: {subset['total_counts'].median():.0f}")
    print(f"  Median mito%: {subset['pct_counts_mt'].median():.2f}%")

print("\n QC calculated!")

In [ ]:
# Apply strict quality filter

median_genes = adata_matchett.obs['n_genes_by_counts'].median()

print(f" Current median genes/cell: {median_genes:.0f}")

if median_genes < 1000:
    print(f"\n  Low quality detected - applying >1000 genes filter...")
    
    adata_filtered = adata_matchett[
        adata_matchett.obs['n_genes_by_counts'] > 1000, :
    ].copy()
    
    print(f"\n   Before: {adata_matchett.n_obs:,} cells")
    print(f"   After: {adata_filtered.n_obs:,} cells")
    print(f"   Removed: {adata_matchett.n_obs - adata_filtered.n_obs:,} cells ({100*(adata_matchett.n_obs - adata_filtered.n_obs)/adata_matchett.n_obs:.1f}%)")
    
    adata_matchett = adata_filtered.copy()
    
    print(f"\n   New median genes: {adata_matchett.obs['n_genes_by_counts'].median():.0f}")
    print(f"   New median UMI: {adata_matchett.obs['total_counts'].median():.0f}")
    print(f"   New median mito%: {adata_matchett.obs['pct_counts_mt'].median():.2f}%")
    
    print(f"\n FILTERED CELLS BY CONDITION:")
    for condition in ['Healthy', 'APAP_ALF']:
        subset = adata_matchett.obs[adata_matchett.obs['condition'] == condition]
        print(f"   {condition}: {len(subset):,} cells")
else:
    print(f" Good quality - no additional filtering needed")

print("\n Quality filtering complete!")

In [ ]:
# Filter for hepatocytes using marker genes (simplified)

print(" Filtering for hepatocytes using marker genes...")

# Human hepatocyte markers
hepatocyte_markers = ['ALB', 'APOA1', 'APOB', 'TTR', 'HP', 'FGA', 'SERPINA1']
available_markers = [m for m in hepatocyte_markers if m in adata_matchett.var_names]

print(f"\n Hepatocyte markers found: {', '.join(available_markers)}")

if available_markers:
    # Calculate hepatocyte score (mean expression of markers)
    # Use raw counts (before normalization in previous failed cell)
    marker_expr = adata_matchett[:, available_markers].X.toarray()
    hepatocyte_score = marker_expr.mean(axis=1)
    
    # Keep cells with hepatocyte score > 0 (expressing at least some markers)
    hepatocyte_filter = hepatocyte_score > 0
    
    print(f"\n Cells expressing hepatocyte markers: {hepatocyte_filter.sum():,} / {len(hepatocyte_filter):,}")
    
    adata_matchett_hep = adata_matchett[hepatocyte_filter, :].copy()
    
    print(f"\n   Before: {adata_matchett.n_obs:,} cells")
    print(f"   After: {adata_matchett_hep.n_obs:,} cells (hepatocytes)")
    print(f"   Removed: {adata_matchett.n_obs - adata_matchett_hep.n_obs:,} non-hepatocyte cells")
    
    adata_matchett = adata_matchett_hep.copy()
else:
    print("\n  No hepatocyte markers found - keeping all cells")

print("\n Hepatocyte filtering complete!")

# Clean up any processing artifacts from failed clustering attempt
if 'highly_variable' in adata_matchett.var.columns:
    del adata_matchett.var['highly_variable']
if 'means' in adata_matchett.var.columns:
    del adata_matchett.var['means']
if 'dispersions' in adata_matchett.var.columns:
    del adata_matchett.var['dispersions']
if 'dispersions_norm' in adata_matchett.var.columns:
    del adata_matchett.var['dispersions_norm']
if 'neighbors' in adata_matchett.uns:
    del adata_matchett.uns['neighbors']
if 'X_pca' in adata_matchett.obsm:
    del adata_matchett.obsm['X_pca']

print(" Cleaned up processing artifacts")

In [ ]:
# Save Matchett Human dataset

output_file = os.path.join(data_processed, "matchett_human_hepatocytes.h5ad")
adata_matchett.write(output_file)

print(f" Dataset saved: {output_file}")
print(f"   File size: {os.path.getsize(output_file) / 1e6:.1f} MB")

print(f"\n Processed dataset:")
print("="*80)
print(f"   Cells: {adata_matchett.n_obs:,}")
print(f"   Genes: {adata_matchett.n_vars:,}")
print(f"   Conditions: {adata_matchett.obs['condition'].unique().tolist()}")
print(f"\n   Quality metrics:")
print(f"   - Median genes/cell: {adata_matchett.obs['n_genes_by_counts'].median():.0f}")
print(f"   - Median UMI/cell: {adata_matchett.obs['total_counts'].median():.0f}")
print(f"   - Median mito %: {adata_matchett.obs['pct_counts_mt'].median():.2f}%")
print(f"\n   Cells by condition:")
for condition in ['Healthy', 'APAP_ALF']:
    count = (adata_matchett.obs['condition'] == condition).sum()
    print(f"   - {condition}: {count:,} cells")
print("="*80)

In [ ]:
# Create summary table

summary_data = []

for condition in ['Healthy', 'APAP_ALF']:
    subset = adata_matchett.obs[adata_matchett.obs['condition'] == condition]
    
    if len(subset) > 0:
        n_samples = subset['sample'].nunique()
        
        summary_data.append({
            'Dataset': 'Matchett_Human',
            'Condition': condition,
            'Cells': len(subset),
            'Samples': n_samples,
            'Median_Genes': int(subset['n_genes_by_counts'].median()),
            'Median_UMI': int(subset['total_counts'].median()),
            'Median_Mito_Pct': f"{subset['pct_counts_mt'].median():.2f}"
        })

summary_df = pd.DataFrame(summary_data)

print(" MATCHETT HUMAN SUMMARY:")
print("="*80)
print(summary_df.to_string(index=False))
print("="*80)

summary_file = os.path.join(data_processed, "matchett_human_summary.csv")
summary_df.to_csv(summary_file, index=False)
print(f"\n Summary saved: {summary_file}")

In [ ]:
# Visualize Matchett Human data

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Plot 1: Cells per condition
condition_counts = adata_matchett.obs['condition'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0, 0].bar(range(len(condition_counts)), condition_counts.values, color=colors, width=0.6)
axes[0, 0].set_xticks(range(len(condition_counts)))
axes[0, 0].set_xticklabels(condition_counts.index, rotation=0)
axes[0, 0].set_ylabel('Cells', fontweight='bold', fontsize=11)
axes[0, 0].set_title('Cells per Condition', fontweight='bold', fontsize=12)
for i, val in enumerate(condition_counts.values):
    axes[0, 0].text(i, val + max(condition_counts.values)*0.02, f'{val:,}', ha='center', fontweight='bold')

# Plot 2: Cells per sample (top 10)
sample_counts = adata_matchett.obs['sample'].value_counts().head(10)
axes[0, 1].barh(range(len(sample_counts)), sample_counts.values, color='#3498db')
axes[0, 1].set_yticks(range(len(sample_counts)))
axes[0, 1].set_yticklabels([s.replace('GSM6964', '') for s in sample_counts.index], fontsize=9)
axes[0, 1].set_xlabel('Cells', fontweight='bold', fontsize=11)
axes[0, 1].set_title('Top 10 Samples by Cell Count', fontweight='bold', fontsize=12)
axes[0, 1].invert_yaxis()

# Plot 3: Genes per cell
axes[0, 2].hist(adata_matchett.obs['n_genes_by_counts'], bins=100, color='#9b59b6', alpha=0.7)
axes[0, 2].axvline(adata_matchett.obs['n_genes_by_counts'].median(), 
                   color='red', linestyle='--', linewidth=2,
                   label=f"Median: {adata_matchett.obs['n_genes_by_counts'].median():.0f}")
axes[0, 2].set_xlabel('Genes per Cell', fontweight='bold', fontsize=11)
axes[0, 2].set_ylabel('Cells', fontweight='bold', fontsize=11)
axes[0, 2].set_title('Genes per Cell', fontweight='bold', fontsize=12)
axes[0, 2].legend()

# Plot 4: UMI per cell
axes[1, 0].hist(adata_matchett.obs['total_counts'], bins=100, color='#e74c3c', alpha=0.7)
axes[1, 0].axvline(adata_matchett.obs['total_counts'].median(), 
                   color='red', linestyle='--', linewidth=2,
                   label=f"Median: {adata_matchett.obs['total_counts'].median():.0f}")
axes[1, 0].set_xlabel('UMI per Cell', fontweight='bold', fontsize=11)
axes[1, 0].set_ylabel('Cells', fontweight='bold', fontsize=11)
axes[1, 0].set_title('UMI per Cell', fontweight='bold', fontsize=12)
axes[1, 0].legend()

# Plot 5: Genes vs UMI by condition
healthy_data = adata_matchett.obs[adata_matchett.obs['condition'] == 'Healthy']
apap_data = adata_matchett.obs[adata_matchett.obs['condition'] == 'APAP_ALF']
axes[1, 1].scatter(healthy_data['total_counts'], healthy_data['n_genes_by_counts'], 
                   s=2, alpha=0.3, c='#2ecc71', label='Healthy')
axes[1, 1].scatter(apap_data['total_counts'], apap_data['n_genes_by_counts'], 
                   s=2, alpha=0.3, c='#e74c3c', label='APAP-ALF')
axes[1, 1].set_xlabel('UMI per Cell', fontweight='bold', fontsize=11)
axes[1, 1].set_ylabel('Genes per Cell', fontweight='bold', fontsize=11)
axes[1, 1].set_title('Genes vs UMI', fontweight='bold', fontsize=12)
axes[1, 1].legend()

# Plot 6: Mito % by condition
axes[1, 2].violinplot([healthy_data['pct_counts_mt'], apap_data['pct_counts_mt']], 
                       positions=[0, 1], showmeans=True, showmedians=True, widths=0.7)
axes[1, 2].set_xticks([0, 1])
axes[1, 2].set_xticklabels(['Healthy', 'APAP-ALF'])
axes[1, 2].set_ylabel('Mitochondrial %', fontweight='bold', fontsize=11)
axes[1, 2].set_title('Mitochondrial % by Condition', fontweight='bold', fontsize=12)

plt.tight_layout()

fig_path = os.path.join(results_path, 'figures/matchett_human_qc.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()

print(f" Figure saved: {fig_path}")